## Part 2: Query Decomposition and Dynamic Filtering

Query decomposition is a necessary component of vector database querying. Mulit part queries that contain context from different clusters in the database will return objects for only some or none of the query components. A query like "show me black formal dresses that aren't in the basics section" has multiple semantic components. No single retrieval call will satisfy it reliably.

In this notebook, we build agents that decompose complex multi-part queries into discrete subqueries and generate dynamic Qdrant filters from natural language. This allows us to generate accurate responses to each semantic component of a complex query AND leverage dynamic filtering to obtain optimum retrieval.

We'll use the `products` collection (2,500 products) from Part 1.

**Key concepts**: query decomposition, subquery generation, dynamic filtering, parallel retrieval, result unification.

## Setup

In [ ]:
# If you installed requirements via pip, do not uncomment this cell
#!pip install pydantic-ai

In [ ]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()

# Setup
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

    def embed_text(text: str) -> list[float]:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    print(f"Qdrant URL: {QDRANT_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

## Reconnect to Qdrant

In [ ]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(url=QDRANT_URL)

COLLECTION_NAME = "products"

# Check our database to make sure our objects are still present
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"Connected. Collection '{COLLECTION_NAME}' has {count} points")

In [ ]:
# Check what payload fields are available
sample = qdrant.scroll(collection_name=COLLECTION_NAME, limit=1)[0]
payload_fields = list(sample[0].payload.keys())
print(f"Payload fields: {payload_fields}")

---

## Introduction to Pydantic AI Agents

Before we build database-aware agents, let's explore what an agent actually is. At its simplest, a Pydantic AI agent is an LLM with **instructions** and optionally **tools**. We create it, give it a task, and it responds.

Let's start with something simple.

### Create a Simple Agent

This agent has one job: answer questions and keep the conversation going by asking a follow-up question.

In [ ]:
from pydantic_ai import Agent

simple_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions="""
    You are a helpful assistant that enjoys conversation.
    You always answer the user's question, then ask a relevant follow-up 
    question to keep the conversation going.
    """,
)

In [ ]:
# Let's see it in action
result = await simple_agent.run("What is the best programming language for data science?")
print(result.output)

### Try It: Create Your Own Agent

Give your agent a personality. Make it a pirate, a Shakespearean actor, a sports commentator — whatever you want. Fill in the instructions and run it.

In [ ]:
my_agent = Agent(
    model="openai:gpt-4.1-mini",
    # TODO: Give your agent some instructions. What personality should it have?
    instructions="""

    """,
)

result = await my_agent.run("What is the meaning of life?")
print(result.output)

In [ ]:
# Let's follow up and answer our agent's question
result = await simple_agent.run("I am so relieved, I though the meaning was deeper than that")
print(result.output)

### The Problem: No Memory


It has no context about our previous question. It doesn't know "which one" refers to programming languages. Each `agent.run()` call is a completely fresh conversation.

### Adding Memory

We can fix this by passing the conversation history from the previous call. Pydantic AI stores all messages (user + assistant) on the result object.

In [ ]:
# First message
result = await simple_agent.run("What is the best programming language for data science?")
print(result.output)
print("\n--- Now with memory ---\n")

In [ ]:
# Follow-up WITH conversation history — now it remembers
result = await simple_agent.run(
    "Which one would you recommend for a beginner?",
    message_history=result.all_messages()
)
print(result.output)

Now it knows we were talking about programming languages and gives a relevant recommendation.

**What we just learned:**
- An **Agent** is an LLM + instructions, that's an agent in its simpliest form.
- Each `agent.run()` is stateless by default and there is no memory between calls.
- Passing `message_history` gives the agent context from previous turns in our conversation.
- Later in Part 3, we'll build a full orchestration system that manages conversation state automatically, follows intent and will reason before it acts.

Now let's connect to our vector database and build agents that can use tools to inspect our data and generate precise database queries no matter how complex!

---

## Build a Dynamic Filtering Agent with Tools

When we created property filters previously, we had to express them programatically. Now, instead of hard-coding valid filter values, we will give the agent a **tool** that lets it inspect the collection at runtime. The agent decides which fields to look up, calls the tool to get the actual values from the database, and then builds precise filters.

This is more robust (works with any dataset) and demonstrates a core concept in the agentic AI REACT framework: **agents  gather information and reason before acting**.

In [ ]:
from pydantic_ai import Agent, Tool
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny

# This is the tool the agent will use to inspect the collection
def get_unique_values(field_name: str) -> str:
    """Look up the unique values for a payload field in the products collection.
    Use this to find the exact values available for filtering before creating filters.
    For example, call this with 'colour_group_name' to see all available colors,
    or 'index_name' to see all available categories."""
    all_points, _ = qdrant.scroll(collection_name=COLLECTION_NAME, limit=2500, with_payload=True)
    values = sorted(set(p.payload.get(field_name, "") for p in all_points if p.payload.get(field_name)))
    return f"Field '{field_name}' has these values: {values}"

# Let's test the tool for ourselves to see what the agent sees
print(get_unique_values("index_name"))
print()
print(get_unique_values("colour_group_name"))

In [ ]:
# Creating our filter agent — same pattern as before, but now with a tool
smart_filter_agent = Agent(
    model="openai:gpt-4.1-mini",
    # These instructions tell the agent HOW to behave and WHAT to do
    # We pass in our payload_fields so it knows what fields exist in our database
    instructions=f"""
    You analyze user queries and create Qdrant filters for an e-commerce product database.
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    For example, if the user says "women's tops", call get_unique_values("index_name") 
    to find that women's products are under "Ladieswear", not "Women" or "Womens".
    
    CRITICAL RULES FOR YOUR RESPONSE:
    - Return ONLY a single Python expression that evaluates to a Filter object
    - Do NOT include import statements, variable assignments, or comments
    - Do NOT wrap your response in markdown code fences
    - The expression must be a single line or a single multi-line expression
    - If no filters needed, return exactly: None
    
    The following classes are already imported and available:
    Filter, FieldCondition, MatchValue, MatchAny
    
    GOOD response examples:
    - Filter(must=[FieldCondition(key="colour_group_name", match=MatchValue(value="Black"))])
    - Filter(must=[FieldCondition(key="index_name", match=MatchValue(value="Ladieswear"))], must_not=[FieldCondition(key="colour_group_name", match=MatchValue(value="Black"))])
    - Filter(must=[FieldCondition(key="index_name", match=MatchValue(value="Menswear")), FieldCondition(key="colour_group_name", match=MatchAny(any=["Dark Blue", "Dark Green", "Dark Grey"]))])
    
    BAD response (DO NOT DO THIS):
    - from qdrant_client.models import Filter ...
    - dark_colors = [...] followed by Filter(...)
    - ```python Filter(...) ```
    """,
    # This is where we give the agent access to our tool
    # The agent decides WHEN to call it based on the user's query
    tools=[Tool(get_unique_values)],
)

# This function ties everything together:
# 1. Send the user's query to the agent
# 2. The agent reasons about what filters to create (calling tools as needed)
# 3. We take the agent's filter code and execute it
# 4. We embed the query and search Qdrant with the generated filters
async def auto_filtered_search(user_query):
    # Step 1: Ask the agent to create filters — it may call get_unique_values behind the scenes
    filter_result = await smart_filter_agent.run(f"Create filters for: {user_query}")
    filter_code = filter_result.output.strip()
    
    # Clean up common LLM formatting issues
    filter_code = filter_code.replace("```python", "").replace("```", "").strip()

    print(f"Query: {user_query}")
    print(f"Generated filter: {filter_code}")

    # Step 2: Try to execute the filter code the agent generated
    query_filter = None
    if filter_code != "None":
        try:
            query_filter = eval(filter_code)
        except Exception as e:
            print(f"Filter creation failed: {e}, searching without filters")

    # Step 3: Embed our query and search with the agent-generated filters
    query_vector = embed_text(user_query)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=5,
    ).points

    # Step 4: Display what we found
    print(f"\nFound {len(results)} products:")
    for i, hit in enumerate(results, 1):
        print(f"  {i}. {hit.payload['prod_name']} ({hit.payload['colour_group_name']}, {hit.payload['section_name']})")

    return results

Our agent has the tools it needs, lets ask it to find a garmet that fits specific properties.

In [ ]:
results = await auto_filtered_search("Show me white tops for women")

Now let's give it soem negative property filters (humans even struggle with these)

In [ ]:
results = await auto_filtered_search("Women's clothing but not black")

## Combining Query Optimization with Dynamic Filtering

Now let's build an agent that cleans up our messy human talk by optimizing the search query AND dynamically generates filters.

Natural language works well with semantic search, however, filler words and unnecessary context can reduce the accuracy of retrieval. LLMs understand vector databases and what words are not needed for retrieval, so we will let our agent create an optimized query for us with filters.

In [ ]:
# This agent does TWO things at once: optimizes the query AND generates filters
# It returns a structured response we parse into separate pieces
combined_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You are an intelligent product search assistant that performs TWO tasks:
    
    1. OPTIMIZE the user's natural language query for better vector search results
    2. GENERATE appropriate Qdrant filters based on the query context
    
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    IMPORTANT FIELD USAGE:
    - For gender/category (men, women, kids): ALWAYS use "index_name" (e.g. "Menswear", "Ladieswear")
    - For color: use "colour_group_name"
    - For product type: use "product_type_name"
    - Do NOT use "department_name" unless the user specifically mentions a department
    
    CRITICAL RULES FOR YOUR FILTER RESPONSE:
    - Return ONLY a single Python expression for the filter, no imports or variables
    - Do NOT wrap in markdown code fences
    
    Return your response in this exact format:
    OPTIMIZED_QUERY: <the optimized search query>
    FILTERS: <single Python Filter expression or None>
    
    Filter syntax:
    - Filter(must=[FieldCondition(key="field", match=MatchValue(value="exact_value"))])
    - Filter(must_not=[FieldCondition(key="field", match=MatchValue(value="exact_value"))])
    - FieldCondition(key="field", match=MatchAny(any=["val1", "val2"])) for multiple values
    """,
    tools=[Tool(get_unique_values)],
)

# This function combines query optimization + dynamic filtering in one flow:
# 1. The agent rewrites the query for better vector search AND generates filters
# 2. We parse the agent's structured response into two parts
# 3. We use both the optimized query and the filters to search Qdrant
async def smart_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    # Step 1: Send to the combined agent — it will optimize the query and create filters
    agent_result = await combined_agent.run(f"Process this query: {user_query}")
    agent_output = agent_result.output
    
    # Step 2: Parse the agent's response — it returns OPTIMIZED_QUERY and FILTERS on separate lines
    lines = agent_output.strip().split('\n')
    optimized_query = None
    filter_code = None
    
    for line in lines:
        if line.startswith('OPTIMIZED_QUERY:'):
            optimized_query = line.replace('OPTIMIZED_QUERY:', '').strip()
        elif line.startswith('FILTERS:'):
            filter_code = line.replace('FILTERS:', '').strip()
    
    # Clean up common LLM formatting issues
    if filter_code:
        filter_code = filter_code.replace("```python", "").replace("```", "").strip()
    
    print(f"Optimized query: '{optimized_query}'")
    print(f"Generated filters: {filter_code}")
    print("=" * 70)
    
    # Step 3: Execute the filter code
    query_filter = None
    if filter_code and filter_code != "None":
        try:
            query_filter = eval(filter_code)
        except Exception as e:
            print(f"Filter generation failed: {e}")
    
    # Step 4: Embed the OPTIMIZED query (not the original) and search with filters
    query_vector = embed_text(optimized_query if optimized_query else user_query)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=5,
    ).points
    
    # Step 5: Display results
    print(f"\nFound {len(results)} products:\n")
    for i, hit in enumerate(results, 1):
        print(f"{i}. {hit.payload['prod_name']}")
        print(f"   Type: {hit.payload['product_type_name']} | Color: {hit.payload['colour_group_name']}")
        print(f"   Section: {hit.payload['section_name']} | Score: {hit.score:.4f}")
        print(f"   {hit.payload['detail_desc'][:120]}...")
        print()
    
    return results

### Example 1: Complex natural language query

In [ ]:
results = await smart_search(
    "I want to see men's dark blue sweaters"
)

### Example 2: Query with implicit filtering needs

In [ ]:
results = await smart_search(
    "I need something nice to wear to a wedding, maybe a dress, nothing too dark"
)

## Query Decomposition Agent

Now let's tackle multi-part queries. A question like "compare the men's jacket options with the women's coat options in dark colors" requires two separate retrieval calls with different filters.

In [ ]:
import json

decomposition_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You decompose complex user queries into discrete subqueries for a product vector database.
    
    Available payload fields: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any field you plan to filter on.
    Do NOT guess values — always check first.
    
    For each subquery, return a JSON array where each element has:
    - "query": the optimized search query for this subquery
    - "filters": Python Qdrant filter code string (or null)
    - "purpose": what this subquery is looking for
    
    Return ONLY valid JSON as your final answer. No explanation.
    
    Qdrant filter syntax:
    - Filter(must=[FieldCondition(key="field", match=MatchValue(value="val"))])
    - FieldCondition(key="field", match=MatchAny(any=["val1", "val2"])) for multiple values
    """,
    tools=[Tool(get_unique_values)],
)

async def decomposed_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    result = await decomposition_agent.run(f"Decompose this query: {user_query}")
    subqueries = json.loads(result.output)
    
    print(f"Decomposed into {len(subqueries)} subqueries:\n")
    
    all_results = []
    for i, sq in enumerate(subqueries, 1):
        print(f"Subquery {i}: {sq['purpose']}")
        print(f"  Search: '{sq['query']}'")
        print(f"  Filters: {sq['filters']}")
        
        query_filter = None
        if sq['filters'] and sq['filters'] != 'null':
            try:
                query_filter = eval(sq['filters'])
            except Exception as e:
                print(f"  Filter failed: {e}")
        
        query_vector = embed_text(sq['query'])
        results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter=query_filter,
            limit=3,
        ).points
        
        print(f"  Found {len(results)} results")
        for hit in results:
            print(f"    - {hit.payload['prod_name']} ({hit.payload['colour_group_name']}, {hit.payload['section_name']})")
        print()
        
        all_results.extend(results)
    
    return all_results

In [ ]:
results = await decomposed_search(
    "Compare men's black sweaters with women's white tops"
)

## Key Takeaways

**What we've built:**
1. **Dynamic filtering agent** — Automatically generates Qdrant payload filters from natural language
2. **Combined optimization + filtering** — Query optimization AND filter generation in one agent
3. **Query decomposition agent** — Breaks multi-part queries into targeted subqueries with individual filters

**The agent advantage:**
- **Before agents**: You'd need to manually write filter code, optimize queries, and chain operations
- **With agents**: Natural language in, optimized database queries with filters out — automatically

Next up: we take this from single-turn RAG to **conversational AI** with intent classification and multi-agent orchestration.

In [ ]:
qdrant.close()